# Module 07: RPA Patterns
## Python Automation as a Conceptual Mirror for Power Automate Desktop

### Learning Objectives

By completing this notebook, you will be able to:

1. Map Power Automate Desktop action groups to their Python equivalents
2. Build a working web scraping automation in Python using `requests` and `BeautifulSoup`
3. Explain the trade-offs between RPA tools and direct API/scripting approaches
4. Apply a decision framework to choose the right automation approach for a given scenario

### Why Python?

Power Automate Desktop abstracts low-level automation into configurable actions. Understanding the Python equivalents reveals what those actions do mechanically — which makes you a better desktop flow designer because you understand the failure modes, the constraints, and the design decisions behind each action group.

### Prerequisites

- Module 07 Guide 01 (Desktop Flows Overview) completed
- Python 3.8+ with `requests` and `beautifulsoup4` installed
- No GUI required — this notebook runs headlessly

### Estimated Time

15 minutes

---

## 1. The Conceptual Map: PAD Actions → Python

Every Power Automate Desktop action group has a Python equivalent. The PAD action is a GUI wrapper around the same underlying mechanism.

| PAD Action Group | Python Equivalent | What Both Do |
|---|---|---|
| **UI Automation** | `pyautogui` | Send keyboard/mouse events to application windows |
| **Web / Browser** | `selenium`, `playwright` | Control a browser via WebDriver protocol |
| **Excel** | `openpyxl`, `xlrd` | Read/write Excel files via COM or file parsing |
| **File System** | `pathlib`, `shutil` | OS-level file and folder operations |
| **Email** | `smtplib`, `imaplib` | SMTP send, IMAP fetch |
| **System** | `subprocess`, `os` | Launch processes, read environment variables |
| **Variables** | Python variables | In-memory state within the automation run |

**Key insight:** PAD actions are not magic — they use the same OS mechanisms as Python scripts. PAD's advantage is a visual designer, built-in error handling UI, cloud integration, and no-code configuration. Python's advantage is unlimited flexibility and direct access to APIs.

---

In [ ]:
# Install required packages (run once)
# Uncomment and run this cell if packages are not already installed.
#
# !pip install requests beautifulsoup4 openpyxl

import requests
from bs4 import BeautifulSoup
import json
import csv
import os
from pathlib import Path
from datetime import datetime, timezone

print("Imports successful.")
print(f"Run timestamp: {datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')}")

## 2. Web Automation: Python vs Power Automate Desktop

### Power Automate Desktop approach (Web action group):

```
1. Launch new Edge (URL: https://books.toscrape.com)
2. Wait for web page content (element: .product_pod)
3. Get details of web page element (CSS: .product_pod h3 a) → %TitleList%
4. Get details of web page element (CSS: .price_color) → %PriceList%
5. Write %TitleList% to Excel worksheet (Sheet1, Column A)
6. Write %PriceList% to Excel worksheet (Sheet1, Column B)
7. Save Excel
8. Close browser
```

### Python equivalent (below):

The Python version uses `requests` to fetch the HTML and `BeautifulSoup` to parse it. This is equivalent to PAD's web actions but faster and more reliable because it does not require a real browser — it reads the raw HTML response directly.

> **When PAD is better:** If the page loads data via JavaScript after initial load (client-side rendering), `requests` cannot see it. PAD's browser actions wait for the full rendered DOM. In those cases, PAD's Web group (or Python's `playwright`/`selenium`) is required.

---

In [ ]:
# Web scraping: extract book titles and prices from a public demo site.
#
# books.toscrape.com is a sandboxed scraping practice site with no real data.
# It is the standard practice target used across Python web scraping courses.
#
# PAD equivalent action sequence:
#   Launch new Edge → Navigate → Wait for content → Get element details (loop)

BASE_URL = "https://books.toscrape.com"

def fetch_page(url: str) -> BeautifulSoup:
    """
    Fetch a URL and return a parsed BeautifulSoup object.

    PAD equivalent: Launch new Edge / Navigate to web page +
                    Wait for web page content

    Returns:
        BeautifulSoup: parsed HTML tree ready for element extraction

    Raises:
        requests.HTTPError: if the server returns a 4xx or 5xx response
    """
    # Set a descriptive User-Agent so the server knows this is a learning exercise.
    headers = {"User-Agent": "Mozilla/5.0 (educational scraper / course demo)"}

    # requests.get sends an HTTP GET — equivalent to the browser navigating to the URL.
    response = requests.get(url, headers=headers, timeout=10)

    # raise_for_status converts 4xx/5xx into exceptions — PAD equivalent is
    # "On block error" catching a navigation failure.
    response.raise_for_status()

    # html.parser is Python's built-in HTML parser. lxml is faster if installed.
    return BeautifulSoup(response.text, "html.parser")


# Fetch the first page
soup = fetch_page(BASE_URL)
print(f"Page title: {soup.title.string}")
print(f"HTTP fetch successful — HTML length: {len(soup.prettify()):,} characters")

### 2.1 Extracting Structured Data

In PAD, you use **Get details of web page element** with a CSS or XPath selector to extract text from a web element. The Python equivalent uses BeautifulSoup's `select()` method with the same CSS selector.

PAD action:
```
Get details of web page element
  UI element: CSS selector = ".product_pod h3 a"
  Attribute: "Own Text"
  Store result in: %BookTitle%
```

Python equivalent: `soup.select(".product_pod h3 a")`

---

In [ ]:
def extract_books(soup: BeautifulSoup) -> list[dict]:
    """
    Extract book title and price from a parsed page.

    PAD equivalent:
        Get details of web page element (title: .product_pod h3 a, attribute: title)
        Get details of web page element (price: .price_color, attribute: Own Text)
        Build a data table row from the two values.

    Returns:
        List of dicts with keys: title, price, rating, availability
    """
    books = []

    # .product_pod is the CSS class on each book card — equivalent to
    # selecting all matching elements on the page.
    for article in soup.select(".product_pod"):

        # The title is in the 'title' attribute of the anchor — not the visible
        # truncated text. PAD's attribute picker lets you choose which attribute
        # to read; here we read 'title' explicitly.
        title_tag = article.select_one("h3 a")
        title = title_tag["title"] if title_tag else "Unknown"

        # Price text looks like "£12.99" — we strip whitespace.
        # PAD equivalent: Get UI element text, then use Replace text action
        # to remove the currency symbol before Convert text to number.
        price_tag = article.select_one(".price_color")
        price_raw = price_tag.get_text(strip=True) if price_tag else "£0.00"

        # Convert "£12.99" → 12.99 (float)
        # PAD equivalent: Replace text (remove '£'), Convert text to number
        price_numeric = float(price_raw.replace("£", "").replace(",", ""))

        # Star rating is encoded as a CSS class word, not text.
        # PAD would need: Get UI element attribute → class → parse the word.
        rating_tag = article.select_one(".star-rating")
        rating_class = rating_tag["class"][1] if rating_tag else "Zero"
        rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
        rating = rating_map.get(rating_class, 0)

        books.append({
            "title": title,
            "price_gbp": price_numeric,
            "rating": rating,
        })

    return books


books = extract_books(soup)

print(f"Extracted {len(books)} books from page 1.")
print()
print("Sample (first 5 books):")
for b in books[:5]:
    print(f"  {b['rating']}* | £{b['price_gbp']:.2f} | {b['title']}")

### 2.2 Multi-Page Extraction (Loop)

PAD handles multi-page scraping with a **Loop** or **Do until** action. The Python equivalent uses a `while` loop.

PAD equivalent:
```
Set %PageNum% = 1
Loop
    Navigate to web page: https://books.toscrape.com/catalogue/page-%PageNum%.html
    Get elements → append to %AllBooks%
    If "next" button not found → break
    Set %PageNum% = %PageNum% + 1
End Loop
```

---

In [ ]:
def scrape_all_pages(base_url: str, max_pages: int = 3) -> list[dict]:
    """
    Scrape multiple pages of books, stopping at max_pages or when
    no 'next' button is found.

    PAD equivalent:
        Loop with page counter, navigate to each page URL,
        check for 'next' button existence to determine if loop continues.

    Args:
        base_url: the site root URL
        max_pages: safety limit to avoid scraping the full 50-page catalog

    Returns:
        List of all book dicts across all pages scraped
    """
    all_books = []
    page_num = 1

    while page_num <= max_pages:
        # Page 1 is at the root; subsequent pages have a catalogue/page-N path.
        if page_num == 1:
            url = base_url
        else:
            url = f"{base_url}/catalogue/page-{page_num}.html"

        print(f"Fetching page {page_num}: {url}")
        page_soup = fetch_page(url)

        page_books = extract_books(page_soup)
        all_books.extend(page_books)
        print(f"  → {len(page_books)} books extracted (running total: {len(all_books)})")

        # Check for the "next" pagination button.
        # PAD equivalent: Wait for UI element with timeout → if not found, break loop.
        next_button = page_soup.select_one("li.next a")
        if next_button is None:
            print("  → No 'next' button found. Stopping.")
            break

        page_num += 1

    return all_books


all_books = scrape_all_pages(BASE_URL, max_pages=3)
print(f"\nTotal books extracted: {len(all_books)}")

## 3. File System Output: Writing Results to CSV

After extracting data, PAD uses its **Excel** or **File** action groups to write results. In Python, `csv.DictWriter` or `openpyxl` does the same work.

PAD equivalent:
```
Launch Excel (new workbook)
For each Book in %AllBooks%
    Write to Excel worksheet: Book.title → Column A, row %CurrentRow%
    Write to Excel worksheet: Book.price_gbp → Column B, row %CurrentRow%
    Write to Excel worksheet: Book.rating → Column C, row %CurrentRow%
    Set %CurrentRow% = %CurrentRow% + 1
End
Save Excel as: books_output.xlsx
Close Excel
```

---

In [ ]:
def write_books_to_csv(books: list[dict], output_path: Path) -> None:
    """
    Write extracted books to a CSV file.

    PAD equivalent:
        Write to Excel worksheet (loop over all_books, write each row)
        Save Excel → Close Excel

    Args:
        books: list of book dicts with title, price_gbp, rating keys
        output_path: Path where the CSV will be written
    """
    # Ensure the output directory exists — PAD equivalent: Create folder if not exists.
    output_path.parent.mkdir(parents=True, exist_ok=True)

    fieldnames = ["title", "price_gbp", "rating"]

    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(books)

    print(f"Written {len(books)} rows to {output_path}")


# Write to a temporary output path.
output_file = Path("/tmp/books_output.csv")
write_books_to_csv(all_books, output_file)

# Verify by reading back the first 3 rows.
print("\nVerification — first 3 rows of output file:")
with open(output_file, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for i, row in enumerate(reader):
        if i >= 3:
            break
        print(f"  {row}")

## 4. RPA Approach Comparison

Three approaches exist for automating repetitive tasks. The right choice depends on what the target system exposes.

| Dimension | Power Automate Desktop (RPA) | Python Scripting | Direct API |
|---|---|---|---|
| **Target** | Any visible UI | Any accessible system | Systems with API endpoints |
| **Requires API?** | No | No (for screen automation) | Yes |
| **Setup time** | Low (record + edit) | Medium (write code) | Medium (learn API) |
| **Maintenance** | High (UI changes break it) | High (site changes break it) | Low (APIs are versioned) |
| **Speed** | Slow (real browser/app) | Fast (headless HTTP) | Very fast (direct calls) |
| **Reliability** | Lower (UI is fragile) | Medium (HTML can change) | High (API contracts are stable) |
| **Skills required** | Low-code (no programming) | Python knowledge | API/HTTP knowledge |
| **Best for** | Legacy apps with no API | Moderate-volume web data | Modern SaaS, high volume |

---

## 5. Decision Framework: RPA vs API Integration

Use this framework to choose the right approach before building anything.

```
Question 1: Does the target system have a public or internal API?
├── YES → Use the API (cloud flow connector or HTTP action)
│         Stop here. APIs are faster, more reliable, and lower maintenance.
│
└── NO  → Question 2: Is the target a web application or a desktop app?
          ├── WEB APP  → Question 3: Does it render content with JavaScript?
          │             ├── NO  → Python requests + BeautifulSoup (fast, headless)
          │             └── YES → PAD Web actions or Playwright (full browser required)
          │
          └── DESKTOP APP → PAD UI Automation actions
                           (only option for thick-client Windows apps)
```

**Secondary considerations once RPA is chosen:**

| If... | Then... |
|---|---||
| Volume > 500 records/run | Build retry logic and error signaling (sentinel values) |
| Run time > 30 minutes | Consider splitting into parallel flows on a machine group |
| Must run overnight without a user | Unattended mode + service account + machine group |
| Target app UI changes frequently | Invest in robust selectors; plan maintenance budget |
| Business unit owns the automation | PAD (no-code); if engineering owns it → Python or Playwright |

---

In [ ]:
def recommend_approach(scenario: dict) -> str:
    """
    Apply the decision framework to a scenario description and return
    the recommended automation approach with reasoning.

    Args:
        scenario: dict with keys:
            has_api (bool): target system exposes a usable API
            is_web (bool): target is a web application (vs desktop app)
            uses_javascript (bool): web content is rendered client-side
            volume_per_run (int): approximate number of records per run
            runs_overnight (bool): must execute without a user present
            owned_by_business (bool): business user (not engineer) maintains it

    Returns:
        str: recommended approach and key considerations
    """
    if scenario["has_api"]:
        return (
            "Approach: Direct API integration (cloud flow connector or HTTP action).\n"
            "Reason: APIs are faster, more reliable, and lower maintenance than RPA.\n"
            "If a Power Automate connector exists, use it. Otherwise, use the HTTP\n"
            "action (Premium) to call the API directly."
        )

    if not scenario["is_web"]:
        approach = "Approach: Power Automate Desktop — UI Automation action group."
        considerations = [
            "Only option for thick-client Windows desktop applications.",
            "Record first, then stabilize selectors by editing AutomationId properties.",
        ]
    elif not scenario["uses_javascript"]:
        approach = "Approach: Python (requests + BeautifulSoup) or PAD Web recorder."
        considerations = [
            "If business users maintain it: use PAD Web recorder (no code required).",
            "If engineers maintain it: Python requests is faster and headless.",
        ]
    else:
        approach = "Approach: Power Automate Desktop — Web action group (full browser)."
        considerations = [
            "JavaScript rendering requires a real browser (Edge or Chrome via PAD).",
            "Consider Playwright (Python) if engineering team prefers code.",
        ]

    # Add volume and overnight considerations.
    if scenario["volume_per_run"] > 500:
        considerations.append(
            f"Volume ({scenario['volume_per_run']} records): split across machine group "
            "for parallel processing."
        )

    if scenario["runs_overnight"]:
        considerations.append(
            "Overnight execution: configure Unattended mode + service account + "
            "machine group for reliability."
        )

    result = approach + "\nConsiderations:\n"
    result += "\n".join(f"  - {c}" for c in considerations)
    return result


# Test three representative scenarios.
scenarios = [
    {
        "name": "Legacy ERP invoice extraction (desktop app, no API)",
        "has_api": False,
        "is_web": False,
        "uses_javascript": False,
        "volume_per_run": 80,
        "runs_overnight": False,
        "owned_by_business": True,
    },
    {
        "name": "Nightly extract from Salesforce (has REST API)",
        "has_api": True,
        "is_web": True,
        "uses_javascript": True,
        "volume_per_run": 2000,
        "runs_overnight": True,
        "owned_by_business": False,
    },
    {
        "name": "Internal portal (React app, no API, 1200 records, overnight)",
        "has_api": False,
        "is_web": True,
        "uses_javascript": True,
        "volume_per_run": 1200,
        "runs_overnight": True,
        "owned_by_business": False,
    },
]

for scenario in scenarios:
    print(f"Scenario: {scenario['name']}")
    print("-" * 60)
    print(recommend_approach(scenario))
    print()

## Exercise 5.1: Apply the Decision Framework

**Task:** Complete the `recommend_approach` calls for the three scenarios below. Run the cell and verify that the output matches your intuition from reading the framework.

Then modify one scenario to flip its recommendation and explain in a comment why the recommendation changed.

**Hint:** Change `has_api` from `False` to `True` on scenario A and observe what happens.

---

In [ ]:
# Exercise 5.1 — Your work here.
#
# Define three scenarios of your own (or use the ones below as a starting point).
# For each, call recommend_approach() and print the result.
# Then modify one scenario to change the recommendation.
#
# Scenario A: Finance team extracts from a web portal (no API, static HTML, 50 records/day)
scenario_a = {
    "name": "Finance web portal extraction",
    "has_api": False,          # Change to True to see recommendation flip
    "is_web": True,
    "uses_javascript": False,  # Static HTML = no JS rendering
    "volume_per_run": 50,
    "runs_overnight": False,
    "owned_by_business": True,
}

# Scenario B: Operations team copies data from old VB6 desktop app into SharePoint
scenario_b = {
    "name": "VB6 desktop app to SharePoint",
    "has_api": False,
    "is_web": False,
    "uses_javascript": False,
    "volume_per_run": 200,
    "runs_overnight": False,
    "owned_by_business": True,
}

# Scenario C: Engineering team needs overnight batch from a React SPA (no API, 800 records)
scenario_c = {
    "name": "React SPA overnight batch",
    "has_api": False,
    "is_web": True,
    "uses_javascript": True,
    "volume_per_run": 800,
    "runs_overnight": True,
    "owned_by_business": False,
}

for s in [scenario_a, scenario_b, scenario_c]:
    print(f"Scenario: {s['name']}")
    print("-" * 60)
    print(recommend_approach(s))
    print()

# Explain your modification here in a comment:
# When I changed scenario_a['has_api'] to True, the recommendation changed from
# [original approach] to Direct API integration because ...

## 6. What `pyautogui` Does (Conceptual — No Execution)

For completeness, here is the Python equivalent of PAD's **UI Automation** action group using `pyautogui`. This code is shown for understanding — do not execute it in this notebook because it requires a real display and would move your actual mouse cursor.

```python
import pyautogui
import time

# PAD equivalent: Click UI element (Customer ID text field)
# pyautogui uses pixel coordinates — fragile if the window moves.
# PAD uses AutomationId — more robust.
pyautogui.click(x=450, y=320)   # Click the Customer ID field
time.sleep(0.5)                  # PAD equivalent: Wait (fixed time — avoid this)

# PAD equivalent: Fill text field
pyautogui.typewrite("C-0001", interval=0.05)

# PAD equivalent: Click UI element (Search button)
pyautogui.click(x=600, y=320)
time.sleep(2)                    # PAD equivalent: Wait for UI element (better: use element detection)

# PAD equivalent: Get UI element attribute (invoice total text)
# pyautogui.screenshot() + image recognition would be needed here —
# PAD's UI Automation reads the accessibility tree directly, which is
# far more reliable than image recognition.
```

**Why PAD is better than `pyautogui` for Windows app automation:**

| Dimension | `pyautogui` | PAD UI Automation |
|---|---|---|
| Element targeting | Pixel coordinates | AutomationId, control type, name |
| Fragility | High (window position changes) | Lower (property-based) |
| Reading values | Requires OCR or image matching | Reads accessibility text directly |
| Setup | `pip install pyautogui` | Visual recorder |
| Debugging | Print statements | Step-through designer with variable inspection |

`pyautogui` is useful for simple, stable workflows or when PAD is not available. For production Windows app automation, PAD's accessibility-tree-based selectors are significantly more reliable.

---

## 7. Summary and Key Takeaways

### What You Built

- A multi-page web scraper using `requests` + `BeautifulSoup` — the Python equivalent of PAD's Web action group
- A CSV writer — the Python equivalent of PAD's Excel/File action groups
- A decision framework function that recommends the right automation approach for any scenario

### Key Takeaways

1. **PAD actions are abstractions over the same mechanisms Python uses** — HTTP, COM automation, file I/O, accessibility trees. Understanding the Python layer makes you a better PAD designer.

2. **The API-first rule is non-negotiable** — if a target system has an API, use it. RPA is a fallback for systems with no API, not a preference.

3. **Choose the tool that fits the team** — business users maintain PAD flows; engineers can use Python/Playwright. The right answer depends on who owns the automation long-term.

4. **Reliability inversely correlates with UI dependency** — APIs > static HTML > JavaScript-rendered HTML > desktop app UI. Plan maintenance budgets accordingly.

5. **Volume and schedule drive the PAD configuration** — low volume + daytime = Attended; high volume + overnight = Unattended + machine group.

### What's Next

The exercise file (`exercises/01_rpa_decision_exercise.py`) gives you 10 automation scenarios to classify using the decision framework. Work through it before Module 08.

### Additional Resources

- [BeautifulSoup documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/) — full CSS selector and traversal reference
- [Power Automate Desktop action reference](https://learn.microsoft.com/en-us/power-automate/desktop-flows/actions-reference) — complete catalog of every PAD action
- [Playwright for Python](https://playwright.dev/python/) — the modern alternative to Selenium for JavaScript-heavy web automation

---

In [ ]:
# Final verification: confirm all notebook outputs are consistent.
#
# This cell runs assertions that verify the earlier cells produced correct results.
# If any assertion fails, re-run the earlier cells in order.

assert len(all_books) > 0, "No books were extracted — check that BASE_URL is reachable."
assert all_books[0]["price_gbp"] > 0, "Price should be a positive number."
assert all_books[0]["rating"] in range(1, 6), "Rating should be 1–5."
assert output_file.exists(), "Output CSV file was not created."

# Verify the CSV has the correct number of rows (header + data rows).
with open(output_file, "r", encoding="utf-8") as f:
    row_count = sum(1 for _ in f) - 1  # subtract header row
assert row_count == len(all_books), (
    f"CSV row count ({row_count}) does not match extracted books ({len(all_books)})."
)

print("All assertions passed.")
print(f"  Books extracted: {len(all_books)}")
print(f"  CSV rows written: {row_count}")
print(f"  Output path: {output_file}")